# Notebook 2: Session Correlation, Warehouse Layer, and Dashboard ETL

This notebook implements the latter parts of the refined data pipeline.

This notebook starts from the cleaned datasets and performs:
1. Session correlation on cleaned event logs.
2. Warehouse-style fact and dimension table creation.
3. Forecast-ready table generation.
4. Dashboard-ready CSV generation.

This reflects the later layers of the refined architecture:
- L3 Session Tracking and Correlation
- Storage / Analytical Warehouse
- Analytics, Visualization, and Business Outputs

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "warehouse_outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

EVENT_FILE = PROCESSED_DIR / "event_logs_cleaned.csv"
TREND_FILE = PROCESSED_DIR / "trend_report_cleaned.csv"
MARKETING_FILE = PROCESSED_DIR / "marketing_summary_cleaned.csv"

print("Reading from processed:", PROCESSED_DIR.resolve())
print("Writing to warehouse output:", OUTPUT_DIR.resolve())

import time

pipeline_start = time.perf_counter()
step_seconds = {}
_lap_t = pipeline_start

def _lap(label):
    """Record elapsed seconds since the previous _lap() call under `label`."""
    global _lap_t
    now = time.perf_counter()
    step_seconds[label] = round(now - _lap_t, 4)
    _lap_t = now
    print(f"  [timing] {label}: {step_seconds[label]}s")


Reading from processed: C:\Users\Lenovo\Downloads\PlatformTech_Finmark_MS2_final\PlatformTech_Finmark-main\finmark_pipeline\data\processed
Writing to warehouse output: C:\Users\Lenovo\Downloads\PlatformTech_Finmark_MS2_final\PlatformTech_Finmark-main\finmark_pipeline\warehouse_outputs


## Step 1: Load the cleaned staging datasets

These files are the outputs of Notebook 1.

Using cleaned files is important because this notebook is meant to represent the later ETL layers,
not the raw cleaning layer. This makes the workflow more accurate and closer to the refined architecture.

In [2]:
event_df = pd.read_csv(EVENT_FILE)
trend_df = pd.read_csv(TREND_FILE)
marketing_df = pd.read_csv(MARKETING_FILE)

print("event_logs_cleaned:", event_df.shape)
print("trend_report_cleaned:", trend_df.shape)
print("marketing_summary_cleaned:", marketing_df.shape)

display(event_df.head(3))
display(trend_df.head(3))
display(marketing_df.head(3))

_lap("load_staging_csvs")


event_logs_cleaned: (2000, 5)
trend_report_cleaned: (20, 3)
marketing_summary_cleaned: (100, 5)


,user_id,event_type,event_time,product_id,amount
0,0b79b857c3765b2e,checkout,2023-06-03 04:13:00,P010,1686.795
1,6e542f6d1814a828,wishlist_add,2023-06-03 05:08:00,P020,2900.630
2,9409c82689e6d63d,profile_update,2023-06-05 06:22:00,P028,1664.010


,week,avg_users,sales_growth_rate
0,2023-W21,328,-0.003
1,2023-W22,280,0.088
2,2023-W23,130,0.073


,date,users_active,total_sales,new_customers,report_generated
0,2023-06-01,179,81287.31,9,2023-06-01 16:00
1,2023-06-02,67,74771.99,5,2023-06-02 16:00
2,2023-06-03,369,84809.74,11,2023-06-03 16:00


  [timing] load_staging_csvs: 0.3206s


## Step 2: Final datatype checks before warehouse transformation

Even though these files are already cleaned, this cell re-checks the critical fields used in warehouse and analytics logic.

In [3]:
event_df["event_time"] = pd.to_datetime(event_df["event_time"], errors="coerce")
event_df["amount"] = pd.to_numeric(event_df["amount"], errors="coerce").fillna(0.0)

trend_df["avg_users"] = pd.to_numeric(trend_df["avg_users"], errors="coerce")
trend_df["sales_growth_rate"] = pd.to_numeric(trend_df["sales_growth_rate"], errors="coerce").fillna(0.0)

marketing_df["date"] = pd.to_datetime(marketing_df["date"], errors="coerce")
marketing_df["report_generated"] = pd.to_datetime(marketing_df["report_generated"], errors="coerce")
marketing_df["users_active"] = pd.to_numeric(marketing_df["users_active"], errors="coerce").fillna(0)
marketing_df["total_sales"] = pd.to_numeric(marketing_df["total_sales"], errors="coerce").fillna(0.0)
marketing_df["new_customers"] = pd.to_numeric(marketing_df["new_customers"], errors="coerce").fillna(0)

# NOTE: row-level validation (missing key columns) and deduplication now
# happen upstream in process.ipynb as L0, before this notebook ever sees
# the data -- no need to repeat it here.

print("Post-validation shapes:")
print("event_df:", event_df.shape)
print("trend_df:", trend_df.shape)
print("marketing_df:", marketing_df.shape)

_lap("type_coercion")


Post-validation shapes:
event_df: (2000, 5)
trend_df: (20, 3)
marketing_df: (100, 5)
  [timing] type_coercion: 0.0605s


## Step 3: Session correlation on cleaned event data

This cell implements the session-correlation layer of the refined architecture.

A session is created by:
- grouping events by user
- sorting the events by event time
- starting a new session when the time gap between events exceeds 30 minutes

The result is a session identifier that can be used for funnel analysis,
customer journey analysis, and session-level KPI reporting.

In [4]:
event_df = event_df.sort_values(["user_id", "event_time"]).copy()

event_df["prev_event_time"] = event_df.groupby("user_id")["event_time"].shift(1)
event_df["minutes_since_prev"] = (
    (event_df["event_time"] - event_df["prev_event_time"]).dt.total_seconds() / 60
)

event_df["new_session_flag"] = np.where(
    event_df["minutes_since_prev"].isna() | (event_df["minutes_since_prev"] > 30),
    1,
    0
)

event_df["session_number"] = event_df.groupby("user_id")["new_session_flag"].cumsum()
event_df["session_id"] = (
    event_df["user_id"].astype(str) + "_S" + event_df["session_number"].astype(str)
)

display(event_df.head(10))

_lap("sessionization")


,user_id,event_type,event_time,product_id,amount,prev_event_time,minutes_since_prev,new_session_flag,session_number,session_id
1388,02445e3b05611b85,add_to_cart,2023-06-02 22:51:00,P019,1771.295,NaT,NaN,1,1,02445e3b05611b85_S1
902,02445e3b05611b85,wishlist_add,2023-06-04 10:26:00,P008,2533.160,2023-06-02 22:51:00,2135.0,1,2,02445e3b05611b85_S2
54,02445e3b05611b85,checkout,2023-06-05 19:54:00,P023,1686.795,2023-06-04 10:26:00,2008.0,1,3,02445e3b05611b85_S3
862,02d7e9358a810d46,page_view,2023-06-02 22:44:00,P007,2722.570,NaT,NaN,1,1,02d7e9358a810d46_S1
598,02d7e9358a810d46,login,2023-06-04 03:14:00,P016,1011.970,2023-06-02 22:44:00,1710.0,1,2,02d7e9358a810d46_S2
541,02d7e9358a810d46,profile_update,2023-06-04 11:37:00,P028,1444.580,2023-06-04 03:14:00,503.0,1,3,02d7e9358a810d46_S3
237,02d7e9358a810d46,checkout,2023-06-06 16:41:00,P020,1686.795,2023-06-04 11:37:00,3184.0,1,4,02d7e9358a810d46_S4
1037,02e8f7b0c33be88e,add_to_cart,2023-06-01 15:42:00,P028,1771.295,NaT,NaN,1,1,02e8f7b0c33be88e_S1
1086,02e8f7b0c33be88e,search,2023-06-01 20:54:00,P007,2777.570,2023-06-01 15:42:00,312.0,1,2,02e8f7b0c33be88e_S2
1750,02e8f7b0c33be88e,profile_update,2023-06-02 15:42:00,P023,1664.010,2023-06-01 20:54:00,1128.0,1,3,02e8f7b0c33be88e_S3


  [timing] sessionization: 0.1318s


## Step 4: Build warehouse dimension tables

Dimension tables describe reusable business entities.
These outputs help transform the staged CSV data into a warehouse-style model.

This notebook creates:
- `dim_users`
- `dim_products`
- `dim_event_types`
- `dim_dates`

These dimensions support downstream joins for dashboard and reporting outputs.

In [5]:
dim_users = event_df[["user_id"]].drop_duplicates().sort_values("user_id").reset_index(drop=True)
dim_users["user_key"] = dim_users.index + 1

dim_products = event_df[["product_id"]].drop_duplicates().sort_values("product_id").reset_index(drop=True)
dim_products["product_key"] = dim_products.index + 1

dim_event_types = event_df[["event_type"]].drop_duplicates().sort_values("event_type").reset_index(drop=True)
dim_event_types["event_type_key"] = dim_event_types.index + 1

all_dates = pd.concat([
    event_df["event_time"].dt.normalize().rename("date"),
    marketing_df["date"].dt.normalize().rename("date")
], ignore_index=True).dropna().drop_duplicates().sort_values().reset_index(drop=True)

dim_dates = pd.DataFrame({"date": all_dates})
dim_dates["date_key"] = dim_dates["date"].dt.strftime("%Y%m%d").astype(int)
dim_dates["year"] = dim_dates["date"].dt.year
dim_dates["month"] = dim_dates["date"].dt.month
dim_dates["day"] = dim_dates["date"].dt.day
dim_dates["day_name"] = dim_dates["date"].dt.day_name()
dim_dates["week_num"] = dim_dates["date"].dt.isocalendar().week.astype(int)

display(dim_users.head())
display(dim_products.head())
display(dim_event_types.head())
display(dim_dates.head())

_lap("build_dimension_tables")


,user_id,user_key
0,02445e3b05611b85,1
1,02d7e9358a810d46,2
2,02e8f7b0c33be88e,3
3,03e994e29c20d135,4
4,0435180e0c345d38,5


,product_id,product_key
0,P001,1
1,P002,2
2,P003,3
3,P004,4
4,P005,5


,event_type,event_type_key
0,add_to_cart,1
1,checkout,2
2,login,3
3,logout,4
4,page_view,5


,date,date_key,year,month,day,day_name,week_num
0,2023-06-01,20230601,2023,6,1,Thursday,22
1,2023-06-02,20230602,2023,6,2,Friday,22
2,2023-06-03,20230603,2023,6,3,Saturday,22
3,2023-06-04,20230604,2023,6,4,Sunday,22
4,2023-06-05,20230605,2023,6,5,Monday,23


  [timing] build_dimension_tables: 0.1356s


## Step 5: Build fact tables for the warehouse layer

Fact tables store measurable business activity.

This notebook creates:
- `fact_events` for event-level activity
- `fact_sessions` for session-level metrics
- `fact_weekly_trends` for weekly trend reporting
- `fact_daily_marketing` for daily business summary metrics

These are the main warehouse tables that support dashboards and analytical reporting.

In [6]:
fact_events = (
    event_df
    .merge(dim_users, on="user_id", how="left")
    .merge(dim_products, on="product_id", how="left")
    .merge(dim_event_types, on="event_type", how="left")
)

fact_events["date"] = fact_events["event_time"].dt.normalize()
fact_events = fact_events.merge(dim_dates[["date", "date_key"]], on="date", how="left")

fact_events = fact_events[
    [
        "user_key", "product_key", "event_type_key", "date_key",
        "session_id", "event_time", "amount", "minutes_since_prev"
    ]
].copy()

fact_sessions = (
    event_df.groupby(["session_id", "user_id"], as_index=False)
    .agg(
        session_start=("event_time", "min"),
        session_end=("event_time", "max"),
        event_count=("event_type", "count"),
        unique_products=("product_id", "nunique"),
        session_revenue=("amount", "sum")
    )
)

fact_sessions["session_duration_minutes"] = (
    (fact_sessions["session_end"] - fact_sessions["session_start"]).dt.total_seconds() / 60
).fillna(0)

fact_sessions = fact_sessions.merge(dim_users, on="user_id", how="left")
fact_sessions["date"] = fact_sessions["session_start"].dt.normalize()
fact_sessions = fact_sessions.merge(dim_dates[["date", "date_key"]], on="date", how="left")

fact_sessions = fact_sessions[
    [
        "session_id", "user_key", "date_key", "session_start", "session_end",
        "session_duration_minutes", "event_count", "unique_products", "session_revenue"
    ]
].copy()

trend_df = trend_df.sort_values("week").reset_index(drop=True).copy()
trend_df["prev_avg_users"] = trend_df["avg_users"].shift(1)
trend_df["prev_sales_growth_rate"] = trend_df["sales_growth_rate"].shift(1)
trend_df["rolling_3wk_avg_users"] = trend_df["avg_users"].rolling(3, min_periods=1).mean()
trend_df["rolling_3wk_growth"] = trend_df["sales_growth_rate"].rolling(3, min_periods=1).mean()
fact_weekly_trends = trend_df.copy()

marketing_df = marketing_df.sort_values("date").reset_index(drop=True).copy()
marketing_df["sales_per_user"] = np.where(
    marketing_df["users_active"] > 0,
    marketing_df["total_sales"] / marketing_df["users_active"],
    0
)
marketing_df["new_customer_rate"] = np.where(
    marketing_df["users_active"] > 0,
    marketing_df["new_customers"] / marketing_df["users_active"],
    0
)
marketing_df["rolling_3day_sales"] = marketing_df["total_sales"].rolling(3, min_periods=1).mean()
marketing_df["rolling_3day_users"] = marketing_df["users_active"].rolling(3, min_periods=1).mean()
marketing_df["rolling_3day_new_customers"] = marketing_df["new_customers"].rolling(3, min_periods=1).mean()
fact_daily_marketing = marketing_df.copy()

display(fact_events.head())
display(fact_sessions.head())
display(fact_weekly_trends.head())
display(fact_daily_marketing.head())

_lap("build_fact_tables")


,user_key,product_key,event_type_key,date_key,session_id,event_time,amount,minutes_since_prev
0,1,19,1,20230602,02445e3b05611b85_S1,2023-06-02 22:51:00,1771.295,NaN
1,1,8,8,20230604,02445e3b05611b85_S2,2023-06-04 10:26:00,2533.160,2135.0
2,1,23,2,20230605,02445e3b05611b85_S3,2023-06-05 19:54:00,1686.795,2008.0
3,2,7,5,20230602,02d7e9358a810d46_S1,2023-06-02 22:44:00,2722.570,NaN
4,2,16,3,20230604,02d7e9358a810d46_S2,2023-06-04 03:14:00,1011.970,1710.0


,session_id,user_key,date_key,session_start,session_end,session_duration_minutes,event_count,unique_products,session_revenue
0,02445e3b05611b85_S1,1,20230602,2023-06-02 22:51:00,2023-06-02 22:51:00,0.0,1,1,1771.295
1,02445e3b05611b85_S2,1,20230604,2023-06-04 10:26:00,2023-06-04 10:26:00,0.0,1,1,2533.160
2,02445e3b05611b85_S3,1,20230605,2023-06-05 19:54:00,2023-06-05 19:54:00,0.0,1,1,1686.795
3,02d7e9358a810d46_S1,2,20230602,2023-06-02 22:44:00,2023-06-02 22:44:00,0.0,1,1,2722.570
4,02d7e9358a810d46_S2,2,20230604,2023-06-04 03:14:00,2023-06-04 03:14:00,0.0,1,1,1011.970


,week,avg_users,sales_growth_rate,prev_avg_users,prev_sales_growth_rate,rolling_3wk_avg_users,rolling_3wk_growth
0,2023-W21,328,-0.003,NaN,NaN,328.000000,-0.003000
1,2023-W22,280,0.088,328.0,-0.003,304.000000,0.042500
2,2023-W23,130,0.073,280.0,0.088,246.000000,0.052667
3,2023-W24,291,0.077,130.0,0.073,233.666667,0.079333
4,2023-W25,200,-0.003,291.0,0.077,207.000000,0.049000


,date,users_active,total_sales,new_customers,report_generated,sales_per_user,new_customer_rate,rolling_3day_sales,rolling_3day_users,rolling_3day_new_customers
0,2023-06-01,179,81287.31,9,2023-06-01 16:00:00,454.119050,0.050279,81287.31,179.000000,9.000000
1,2023-06-02,67,74771.99,5,2023-06-02 16:00:00,1115.999851,0.074627,78029.65,123.000000,7.000000
2,2023-06-03,369,84809.74,11,2023-06-03 16:00:00,229.836694,0.029810,80289.68,205.000000,8.333333
3,2023-06-04,94,61212.30,3,2023-06-04 16:00:00,651.194681,0.031915,73598.01,176.666667,6.333333
4,2023-06-05,402,80911.49,10,2023-06-05 16:00:00,201.272363,0.024876,75644.51,288.333333,8.000000


  [timing] build_fact_tables: 0.2382s


## Step 6: Create forecast-ready ETL outputs

These tables are forecast-ready analytical datasets that can feed forecasting logic later.

Each source is transformed individually:
1. Event logs -> daily event trend forecast base
2. Trend report -> weekly trend forecast base
3. Marketing summary -> daily business KPI forecast base

In [7]:
forecast_event_logs = (
    event_df.assign(event_date=event_df["event_time"].dt.normalize())
    .groupby("event_date", as_index=False)
    .agg(
        total_events=("event_type", "count"),
        unique_users=("user_id", "nunique"),
        unique_products=("product_id", "nunique"),
        checkout_revenue=("amount", "sum")
    )
    .sort_values("event_date")
)

forecast_event_logs["rolling_3day_events"] = forecast_event_logs["total_events"].rolling(3, min_periods=1).mean()
forecast_event_logs["rolling_3day_users"] = forecast_event_logs["unique_users"].rolling(3, min_periods=1).mean()
forecast_event_logs["rolling_3day_revenue"] = forecast_event_logs["checkout_revenue"].rolling(3, min_periods=1).mean()

forecast_trend_report = fact_weekly_trends[
    [
        "week", "avg_users", "sales_growth_rate",
        "prev_avg_users", "prev_sales_growth_rate",
        "rolling_3wk_avg_users", "rolling_3wk_growth"
    ]
].copy()

forecast_marketing_summary = fact_daily_marketing[
    [
        "date", "users_active", "total_sales", "new_customers",
        "sales_per_user", "new_customer_rate",
        "rolling_3day_sales", "rolling_3day_users", "rolling_3day_new_customers"
    ]
].copy()

display(forecast_event_logs.head())
display(forecast_trend_report.head())
display(forecast_marketing_summary.head())

_lap("build_forecast_outputs")


,event_date,total_events,unique_users,unique_products,checkout_revenue,rolling_3day_events,rolling_3day_users,rolling_3day_revenue
0,2023-06-01,223,166,29,373689.445,223.000000,166.000000,373689.445000
1,2023-06-02,343,246,30,559524.370,283.000000,206.000000,466606.907500
2,2023-06-03,380,241,30,603186.680,315.333333,217.666667,512133.498333
3,2023-06-04,378,243,30,600121.520,367.000000,243.333333,587610.856667
4,2023-06-05,377,245,30,609021.905,378.333333,243.000000,604110.035000


,week,avg_users,sales_growth_rate,prev_avg_users,prev_sales_growth_rate,rolling_3wk_avg_users,rolling_3wk_growth
0,2023-W21,328,-0.003,NaN,NaN,328.000000,-0.003000
1,2023-W22,280,0.088,328.0,-0.003,304.000000,0.042500
2,2023-W23,130,0.073,280.0,0.088,246.000000,0.052667
3,2023-W24,291,0.077,130.0,0.073,233.666667,0.079333
4,2023-W25,200,-0.003,291.0,0.077,207.000000,0.049000


,date,users_active,total_sales,new_customers,sales_per_user,new_customer_rate,rolling_3day_sales,rolling_3day_users,rolling_3day_new_customers
0,2023-06-01,179,81287.31,9,454.119050,0.050279,81287.31,179.000000,9.000000
1,2023-06-02,67,74771.99,5,1115.999851,0.074627,78029.65,123.000000,7.000000
2,2023-06-03,369,84809.74,11,229.836694,0.029810,80289.68,205.000000,8.333333
3,2023-06-04,94,61212.30,3,651.194681,0.031915,73598.01,176.666667,6.333333
4,2023-06-05,402,80911.49,10,201.272363,0.024876,75644.51,288.333333,8.000000


  [timing] build_forecast_outputs: 0.136s


## Step 7: Create dashboard-ready curated outputs

These outputs represent the production dashboard layer in simplified CSV form.

They are designed to be easy inputs for Power BI, Tableau, or similar visualization tools.
The idea is that the warehouse fact tables are still detailed, while these dashboard tables are already business-facing.

In [8]:
dashboard_event_type = (
    event_df.groupby("event_type", as_index=False)
    .agg(
        event_count=("event_type", "count"),
        unique_users=("user_id", "nunique"),
        total_amount=("amount", "sum")
    )
    .sort_values("event_count", ascending=False)
)

dashboard_daily_kpi = fact_daily_marketing[
    ["date", "users_active", "total_sales", "new_customers", "sales_per_user", "new_customer_rate"]
].copy()

dashboard_session_summary = (
    fact_sessions.groupby("date_key", as_index=False)
    .agg(
        total_sessions=("session_id", "count"),
        avg_session_duration_minutes=("session_duration_minutes", "mean"),
        total_session_revenue=("session_revenue", "sum"),
        avg_events_per_session=("event_count", "mean")
    )
    .sort_values("date_key")
)

dashboard_master_kpi = (
    dashboard_daily_kpi.merge(
        dashboard_session_summary,
        on=None,
        how="cross"
    )
)

display(dashboard_event_type.head())
display(dashboard_daily_kpi.head())
display(dashboard_session_summary.head())

_lap("build_dashboard_outputs")


,event_type,event_count,unique_users,total_amount
2,login,276,200,409583.89
1,checkout,260,190,428039.13
7,wishlist_add,260,196,435730.70
5,profile_update,255,190,424684.38
0,add_to_cart,252,178,438762.07


,date,users_active,total_sales,new_customers,sales_per_user,new_customer_rate
0,2023-06-01,179,81287.31,9,454.119050,0.050279
1,2023-06-02,67,74771.99,5,1115.999851,0.074627
2,2023-06-03,369,84809.74,11,229.836694,0.029810
3,2023-06-04,94,61212.30,3,651.194681,0.031915
4,2023-06-05,402,80911.49,10,201.272363,0.024876


,date_key,total_sessions,avg_session_duration_minutes,total_session_revenue,avg_events_per_session
0,20230601,221,0.049774,373689.445,1.009050
1,20230602,338,0.272189,559524.370,1.014793
2,20230603,370,0.337838,603186.680,1.027027
3,20230604,370,0.391892,600121.520,1.021622
4,20230605,373,0.179625,609021.905,1.010724


  [timing] build_dashboard_outputs: 0.3017s


## Step 8: Optional simple forecast columns

This cell adds a very simple baseline forecast using rolling averages.

This is useful for demonstrating that the ETL outputs are forecast-capable.

In [9]:
forecast_event_logs["next_day_events_baseline"] = forecast_event_logs["rolling_3day_events"].shift(1)
forecast_event_logs["next_day_revenue_baseline"] = forecast_event_logs["rolling_3day_revenue"].shift(1)

forecast_marketing_summary["next_day_sales_baseline"] = forecast_marketing_summary["rolling_3day_sales"].shift(1)
forecast_marketing_summary["next_day_users_baseline"] = forecast_marketing_summary["rolling_3day_users"].shift(1)
forecast_marketing_summary["next_day_new_customers_baseline"] = forecast_marketing_summary["rolling_3day_new_customers"].shift(1)

forecast_trend_report["next_week_avg_users_baseline"] = forecast_trend_report["rolling_3wk_avg_users"].shift(1)
forecast_trend_report["next_week_growth_baseline"] = forecast_trend_report["rolling_3wk_growth"].shift(1)

display(forecast_event_logs.head())
display(forecast_marketing_summary.head())
display(forecast_trend_report.head())

_lap("add_baseline_forecast_columns")


,event_date,total_events,unique_users,unique_products,checkout_revenue,rolling_3day_events,rolling_3day_users,rolling_3day_revenue,next_day_events_baseline,next_day_revenue_baseline
0,2023-06-01,223,166,29,373689.445,223.000000,166.000000,373689.445000,NaN,NaN
1,2023-06-02,343,246,30,559524.370,283.000000,206.000000,466606.907500,223.000000,373689.445000
2,2023-06-03,380,241,30,603186.680,315.333333,217.666667,512133.498333,283.000000,466606.907500
3,2023-06-04,378,243,30,600121.520,367.000000,243.333333,587610.856667,315.333333,512133.498333
4,2023-06-05,377,245,30,609021.905,378.333333,243.000000,604110.035000,367.000000,587610.856667


,date,users_active,total_sales,new_customers,sales_per_user,new_customer_rate,rolling_3day_sales,rolling_3day_users,rolling_3day_new_customers,next_day_sales_baseline,next_day_users_baseline,next_day_new_customers_baseline
0,2023-06-01,179,81287.31,9,454.119050,0.050279,81287.31,179.000000,9.000000,NaN,NaN,NaN
1,2023-06-02,67,74771.99,5,1115.999851,0.074627,78029.65,123.000000,7.000000,81287.31,179.000000,9.000000
2,2023-06-03,369,84809.74,11,229.836694,0.029810,80289.68,205.000000,8.333333,78029.65,123.000000,7.000000
3,2023-06-04,94,61212.30,3,651.194681,0.031915,73598.01,176.666667,6.333333,80289.68,205.000000,8.333333
4,2023-06-05,402,80911.49,10,201.272363,0.024876,75644.51,288.333333,8.000000,73598.01,176.666667,6.333333


,week,avg_users,sales_growth_rate,prev_avg_users,prev_sales_growth_rate,rolling_3wk_avg_users,rolling_3wk_growth,next_week_avg_users_baseline,next_week_growth_baseline
0,2023-W21,328,-0.003,NaN,NaN,328.000000,-0.003000,NaN,NaN
1,2023-W22,280,0.088,328.0,-0.003,304.000000,0.042500,328.000000,-0.003000
2,2023-W23,130,0.073,280.0,0.088,246.000000,0.052667,304.000000,0.042500
3,2023-W24,291,0.077,130.0,0.073,233.666667,0.079333,246.000000,0.052667
4,2023-W25,200,-0.003,291.0,0.077,207.000000,0.049000,233.666667,0.079333


  [timing] add_baseline_forecast_columns: 0.1461s


## Step 9: Export warehouse, forecast, and dashboard CSVs

This is the final ETL output step.
The notebook writes curated CSVs representing the warehouse layer and production dashboard layer.

In [10]:
from pathlib import Path

BASE_DIR = Path(".")
WAREHOUSE_DIR = BASE_DIR / "warehouse_outputs"
FINAL_DIR = BASE_DIR / "data" / "final_visuals"

WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "dim_users.csv": dim_users,
    "dim_products.csv": dim_products,
    "dim_event_types.csv": dim_event_types,
    "dim_dates.csv": dim_dates,
    "fact_events.csv": fact_events,
    "fact_sessions.csv": fact_sessions,
    "fact_weekly_trends.csv": fact_weekly_trends,
    "fact_daily_marketing.csv": fact_daily_marketing,
    "forecast_event_logs.csv": forecast_event_logs,
    "forecast_trend_report.csv": forecast_trend_report,
    "forecast_marketing_summary.csv": forecast_marketing_summary,
}

dashboard_exports = {
    "dashboard_event_type.csv": dashboard_event_type,
    "dashboard_daily_kpi.csv": dashboard_daily_kpi,
    "dashboard_session_summary.csv": dashboard_session_summary,
}

for filename, df in exports.items():
    df.to_csv(WAREHOUSE_DIR / filename, index=False)

for filename, df in dashboard_exports.items():
    df.to_csv(FINAL_DIR / filename, index=False)

print("Exported warehouse files:")
for filename in exports:
    print("-", filename)

print("\nExported final visualization files:")
for filename in dashboard_exports:
    print("-", filename)

_lap("export_csv")

pipeline_total_seconds = round(time.perf_counter() - pipeline_start, 4)
step_seconds["pipeline_total_seconds"] = pipeline_total_seconds

report_path = WAREHOUSE_DIR / "transformation_run_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "run_timestamp": pd.Timestamp.now().isoformat(),
            "input_rows": {
                "event_df": int(event_df.shape[0]),
                "trend_df": int(trend_df.shape[0]),
                "marketing_df": int(marketing_df.shape[0]),
            },
            "step_seconds": step_seconds,
        },
        f,
        indent=4,
    )

print(f"\nFull transformation pipeline runtime: {pipeline_total_seconds}s")
print("Timing report saved to:", report_path)


Exported warehouse files:
- dim_users.csv
- dim_products.csv
- dim_event_types.csv
- dim_dates.csv
- fact_events.csv
- fact_sessions.csv
- fact_weekly_trends.csv
- fact_daily_marketing.csv
- forecast_event_logs.csv
- forecast_trend_report.csv
- forecast_marketing_summary.csv

Exported final visualization files:
- dashboard_event_type.csv
- dashboard_daily_kpi.csv
- dashboard_session_summary.csv
  [timing] export_csv: 0.2261s

Full transformation pipeline runtime: 1.6968s
Timing report saved to: warehouse_outputs\transformation_run_report.json


## Step 10: Final validation

This final cell confirms that the warehouse and dashboard ETL outputs were created successfully.

At this stage:
- session correlation has been completed
- warehouse-style fact and dimension tables now exist
- forecast-ready CSV outputs are available
- dashboard-ready CSV outputs are available

In [11]:
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(path.name, "-", path.stat().st_size, "bytes")

print("\nPreview: fact_sessions")
display(fact_sessions.head())

print("\nPreview: forecast_event_logs")
display(forecast_event_logs.head())

print("\nPreview: forecast_trend_report")
display(forecast_trend_report.head())

print("\nPreview: forecast_marketing_summary")
display(forecast_marketing_summary.head())

dim_dates.csv - 4227 bytes
dim_event_types.csv - 130 bytes
dim_products.csv - 285 bytes
dim_users.csv - 8710 bytes
fact_daily_marketing.csv - 13067 bytes
fact_events.csv - 143675 bytes
fact_sessions.csv - 176979 bytes
fact_weekly_trends.csv - 1393 bytes
forecast_event_logs.csv - 745 bytes
forecast_marketing_summary.csv - 15516 bytes
forecast_trend_report.csv - 2061 bytes

Preview: fact_sessions


,session_id,user_key,date_key,session_start,session_end,session_duration_minutes,event_count,unique_products,session_revenue
0,02445e3b05611b85_S1,1,20230602,2023-06-02 22:51:00,2023-06-02 22:51:00,0.0,1,1,1771.295
1,02445e3b05611b85_S2,1,20230604,2023-06-04 10:26:00,2023-06-04 10:26:00,0.0,1,1,2533.160
2,02445e3b05611b85_S3,1,20230605,2023-06-05 19:54:00,2023-06-05 19:54:00,0.0,1,1,1686.795
3,02d7e9358a810d46_S1,2,20230602,2023-06-02 22:44:00,2023-06-02 22:44:00,0.0,1,1,2722.570
4,02d7e9358a810d46_S2,2,20230604,2023-06-04 03:14:00,2023-06-04 03:14:00,0.0,1,1,1011.970



Preview: forecast_event_logs


,event_date,total_events,unique_users,unique_products,checkout_revenue,rolling_3day_events,rolling_3day_users,rolling_3day_revenue,next_day_events_baseline,next_day_revenue_baseline
0,2023-06-01,223,166,29,373689.445,223.000000,166.000000,373689.445000,NaN,NaN
1,2023-06-02,343,246,30,559524.370,283.000000,206.000000,466606.907500,223.000000,373689.445000
2,2023-06-03,380,241,30,603186.680,315.333333,217.666667,512133.498333,283.000000,466606.907500
3,2023-06-04,378,243,30,600121.520,367.000000,243.333333,587610.856667,315.333333,512133.498333
4,2023-06-05,377,245,30,609021.905,378.333333,243.000000,604110.035000,367.000000,587610.856667



Preview: forecast_trend_report


,week,avg_users,sales_growth_rate,prev_avg_users,prev_sales_growth_rate,rolling_3wk_avg_users,rolling_3wk_growth,next_week_avg_users_baseline,next_week_growth_baseline
0,2023-W21,328,-0.003,NaN,NaN,328.000000,-0.003000,NaN,NaN
1,2023-W22,280,0.088,328.0,-0.003,304.000000,0.042500,328.000000,-0.003000
2,2023-W23,130,0.073,280.0,0.088,246.000000,0.052667,304.000000,0.042500
3,2023-W24,291,0.077,130.0,0.073,233.666667,0.079333,246.000000,0.052667
4,2023-W25,200,-0.003,291.0,0.077,207.000000,0.049000,233.666667,0.079333



Preview: forecast_marketing_summary


,date,users_active,total_sales,new_customers,sales_per_user,new_customer_rate,rolling_3day_sales,rolling_3day_users,rolling_3day_new_customers,next_day_sales_baseline,next_day_users_baseline,next_day_new_customers_baseline
0,2023-06-01,179,81287.31,9,454.119050,0.050279,81287.31,179.000000,9.000000,NaN,NaN,NaN
1,2023-06-02,67,74771.99,5,1115.999851,0.074627,78029.65,123.000000,7.000000,81287.31,179.000000,9.000000
2,2023-06-03,369,84809.74,11,229.836694,0.029810,80289.68,205.000000,8.333333,78029.65,123.000000,7.000000
3,2023-06-04,94,61212.30,3,651.194681,0.031915,73598.01,176.666667,6.333333,80289.68,205.000000,8.333333
4,2023-06-05,402,80911.49,10,201.272363,0.024876,75644.51,288.333333,8.000000,73598.01,176.666667,6.333333
